<a href="https://colab.research.google.com/github/OrastaRakhmatullayeva/ML-projects/blob/main/LMST_full_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch, torch.nn as nn, torch.nn.functional as F


In [2]:
lstm=nn.LSTM(3,8,batch_first=True)
x=torch.randn(4,10,3)
out,(hn,cn)=lstm(x)
print(out.shape,hn.shape,cn.shape)

torch.Size([4, 10, 8]) torch.Size([1, 4, 8]) torch.Size([1, 4, 8])


In [3]:
gru=nn.GRU(3,8,batch_first=True)
x=torch.randn(4,10,3)
out,hn=gru(x)
print(out.shape,hn.shape)

torch.Size([4, 10, 8]) torch.Size([1, 4, 8])


In [4]:
for name, kind in (("RNN", nn.RNN), ("LSTM", nn.LSTM), ("GRU", nn.GRU)):
    print(name, sum(p.numel() for p in kind(1, 16).parameters()))


RNN 304
LSTM 1216
GRU 912


In [5]:
import pandas as pd

In [6]:
import pandas as pd
url = ("https://raw.githubusercontent.com/justmarkham/"
       "pycon-2016-tutorial/master/data/sms.tsv")
df=pd.read_csv(url,sep='\t',names=["label",'text'])
print(df.shape,df.label.value_counts())
print(df.text[1])

(5572, 2) label
ham     4825
spam     747
Name: count, dtype: int64
Ok lar... Joking wif u oni...


In [7]:
df.head()

,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


1.Datalarni tozalash va tokenlash

In [8]:
import re
def tokensize(text):
  return re.findall(r"[a-z0-9]+",text.lower())  # ortiqcha narsalarni olib tashlab, faqat harflardan va raqamlardan iborat qismini oladi. Kichik harflar bilan qaytaradi


print(tokensize(df.text[1]))
print(tokensize("WINNER! CALL 0080 and claim your prize!"))



['ok', 'lar', 'joking', 'wif', 'u', 'oni']
['winner', 'call', '0080', 'and', 'claim', 'your', 'prize']


2. To pass dictionary


In [9]:
from collections import Counter
from sklearn.model_selection import train_test_split
tr,va=train_test_split(df,test_size=0.2,stratify=df.label,random_state=42)

counts=Counter(w for t in tr.text for w in tokensize(t))
vocab={"<pad>":0,"<unk>":1}

for w , c in counts.items():
  if c>=2:
    vocab[w]=len(vocab)
print(len(tr),len(va),len(vocab))
print(counts.most_common(4))

4457 1115 3743
[('i', 2409), ('to', 1787), ('you', 1760), ('a', 1149)]


3. Index and Padding

In [10]:
def encode(text,maxlen=30):
  ids=[vocab.get(w,1) for w in tokensize(text)][:maxlen]
  return [0]*(maxlen-len(ids))+ids

print(encode("ok,hello my name is",10))

[0, 0, 0, 0, 0, 21, 829, 69, 527, 83]


To separate the data of val,train, test


In [28]:
X_tr=torch.tensor([encode(t) for t in tr.text])
X_va=torch.tensor([encode(t) for t in va.text])
y_tr=torch.tensor((tr.label=='spam').values).float()
y_va=torch.tensor((va.label=='spam').values).float()

print(X_tr.shape,round(y_tr.mean().item(),3))

torch.Size([4457, 30]) 0.134


4.Embedding

In [29]:
class Spam(nn.Module):
  def __init__(self,kind=nn.LSTM):
    super().__init__()
    self.emb=nn.Embedding(len(vocab),32,padding_idx=32)
    self.rnn=kind(32,32,batch_first=True)
    self.fc=nn.Linear(32,1)
  def forward(self,x):
    out,_=self.rnn(self.emb(x))
    return self.fc(out[:,1].squeeze(1)) # eliminate the columns with one dimension

print(Spam()(X_tr[:5]).shape)

torch.Size([5, 1])


In [30]:
def fit(model, epochs=5):
    opt = torch.optim.Adam(model.parameters(), lr=0.005)
    for _ in range(epochs):
        for xb, yb in loader:
            out = model(xb).squeeze(1)                       # [64, 1] -> [64]
            loss = F.binary_cross_entropy_with_logits(out, yb.float())
            opt.zero_grad(); loss.backward(); opt.step()

    with torch.no_grad():
        pred = (model(X_va).squeeze(1) > 0).float()          # [N, 1] -> [N]
    return round(f1_score(y_va, pred), 3)

In [31]:
for name, kind in (("RNN", nn.RNN), ("LSTM", nn.LSTM), ("GRU", nn.GRU)):
    res = []
    for sd in range(3):
        torch.manual_seed(sd)
        res.append(fit(Spam(kind)))
    print(name, res)

RNN [0.304, 0.304, 0.31]
LSTM [0.33, 0.323, 0.321]
GRU [0.325, 0.294, 0.314]


In [32]:
torch.manual_seed(0)
model = Spam(nn.GRU)
fit(model)

msgs = ["“CONGRATULATIONS! You have been selected as the WINNER of a $1,000 Walmart gift card! Click here NOW to claim your FREE prize before it expires: http://bit.ly/claim-free-prize. Limited time offer, act fast! Reply STOP to unsubscribe.”",
        "ok see you at free lunch tomorrow"]

for t in msgs:
    p = torch.sigmoid(model(torch.tensor([encode(t)])))
    print(round(p.item(), 3), t)

0.662 “CONGRATULATIONS! You have been selected as the WINNER of a $1,000 Walmart gift card! Click here NOW to claim your FREE prize before it expires: http://bit.ly/claim-free-prize. Limited time offer, act fast! Reply STOP to unsubscribe.”
0.081 ok see you at free lunch tomorrow
